# Explore and Discover

This notebook is your playground. You have already built a complete galaxy modelling pipeline — now we will use it to explore real physics.

Each experiment below asks you to change something and observe what happens. There are guiding questions, but **do not just answer them and move on** — try to understand *why* you see what you see.

---

## Setup

Run this cell first — it imports everything and re-uses the galaxy you built in Notebook 4.

In [ ]:
# Run this cell first — it installs everything you need.
# On COSMA or if packages are already installed this will be quick.

%pip install -q cosmos-synthesizer astropy h5py requests
print("All packages ready!")

In [ ]:
import sys, os
sys.path.insert(0, ".")
from helpers import (
    download_tng_galaxy, load_tng_particles,
    build_synthesizer_galaxy, get_spectrum,
    make_image, plot_spectrum, plot_one_image,
    plot_rgb, plot_particles,
)

import numpy as np
import matplotlib.pyplot as plt
from synthesizer.instruments import FilterCollection

# ============================================================
# PASTE YOUR API KEY HERE AGAIN
# ============================================================
API_KEY = "YOUR_API_KEY_HERE"
# ============================================================

print("Ready! Starting with Galaxy 553837 — run Notebook 4 first if you haven't already.")

In [ ]:
# Load and build the default galaxy (same as Notebook 4)
# If you already ran Notebook 4 this will be fast (data is cached)

GALAXY_ID = 553837

data_file    = download_tng_galaxy(GALAXY_ID, api_key=API_KEY)
particles    = load_tng_particles(data_file)
galaxy, grid = build_synthesizer_galaxy(
    particles,
    grid_dir  = "../grids",
    grid_name = "maraston24-Te00_kroupa-0.1,100"
)

print("\nRunning Synthesizer...")
sed, spectrum_label = get_spectrum(galaxy, grid)
print("Done!")

---

## Experiment 1 — What Different Wavelengths Reveal

Now let's compare the same galaxy across a wide range of wavelengths, from the far ultraviolet to the near-infrared.

Remember:
- **UV light** comes from **hot, young, massive stars** — galaxies that are actively forming stars
- **Red/infrared light** comes from **cool, old, low-mass stars** — galaxies with an established older population
- **The distribution** of light tells you about the galaxy's **star formation history**

In [ ]:
# Make images of galaxy 553837 in a range of filters

from synthesizer.instruments import FilterCollection, Instrument
from unyt import kpc as kpc_unit

bands = [
    ("GALEX/GALEX.FUV",   "purple",  "Far UV (~1540 Å)"),
    ("GALEX/GALEX.NUV",   "violet",  "Near UV (~2300 Å)"),
    ("SLOAN/SLOAN.u",     "blue",    "u-band (~3550 Å)"),
    ("SLOAN/SLOAN.g",     "green",   "g-band (~4770 Å)"),
    ("SLOAN/SLOAN.r",     "orange",  "r-band (~6230 Å)"),
    ("SLOAN/SLOAN.i",     "red",     "i-band (~7630 Å)"),
    ("2MASS/2MASS.J",     "darkred", "J-band (~1.2 μm)"),
    ("2MASS/2MASS.K",     "maroon",  "K-band (~2.2 μm)"),
]

all_filter_codes = [b[0] for b in bands]

print("Making images in 8 filters (this will take a few minutes)...")
all_images = make_image(
    galaxy, spectrum_label,
    fov_kpc=60, pixel_kpc=0.6,
    filter_codes=all_filter_codes
)
print("Done!")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9), facecolor='black')
axes = axes.flatten()

for ax, (code, colour, label) in zip(axes, bands):
    arr = np.array(all_images[code].arr)
    arr_log = np.where(arr > 0, np.log10(arr + 1e-40), np.nan)
    vmin = np.nanpercentile(arr_log, 5)
    vmax = np.nanpercentile(arr_log, 99.5)
    ax.imshow(arr_log, origin='lower', cmap='magma', vmin=vmin, vmax=vmax)
    ax.set_title(label, color=colour, fontsize=9.5, pad=4)
    ax.axis('off')
    ax.set_facecolor('black')

fig.patch.set_facecolor('black')
fig.suptitle(f"Galaxy {GALAXY_ID} — UV to Infrared",
             color='white', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### Questions for Experiment 1

1. Compare the **UV image** with the **K-band (infrared) image**. Are they the same shape? If not, why not?
   *Hint: think about where young stars form in a spiral galaxy (outer arms) vs where old stars live (the bulge in the centre).*

2. If you could only observe this galaxy through one filter, which would you choose to best measure its **total stellar mass** (all the stars it has ever made)?
   *Hint: infrared light is less affected by young massive stars and dust.*

3. If you could only observe through one filter to measure the **current star formation rate** (how many new stars are forming right now), which would you choose?

---

## Experiment 2 — What If There Were No Dust?

Dust absorbs starlight — especially UV. We can test the effect of dust by looking at the spectrum with and without it.

In this experiment we use the **Calzetti attenuation law** — a standard model for how dust affects galaxy spectra. The key parameter is **A_V**: the amount of attenuation at the V-band (5500 Å).

- A_V = 0 → no dust
- A_V = 1 → about 60% of V-band light is absorbed
- A_V = 3 → about 94% of V-band light is absorbed

In [ ]:
# Apply different amounts of dust attenuation to the spectrum

from synthesizer.emission_models.attenuation import CalzettiAtt
from synthesizer.emission_models import AttenuatedEmission, StellarEmissionModel

def get_attenuated_spectrum(galaxy, grid, A_V):
    """
    Return the galaxy spectrum attenuated by dust with V-band attenuation A_V.
    A_V = 0 means no dust.
    """
    from unyt import dimensionless
    stellar_model = StellarEmissionModel(grid)
    dust_model    = CalzettiAtt(A_V=A_V)
    att_model     = AttenuatedEmission(emitter=stellar_model, attenuation=dust_model)
    galaxy.get_spectra(att_model)
    available = list(galaxy.stars.spectra.keys())
    # Attenuated spectrum is typically labelled 'attenuated'
    for key in available:
        if 'attenuated' in key.lower() or 'dust' in key.lower():
            return galaxy.stars.spectra[key]
    # Fallback: return the last generated one
    return galaxy.stars.spectra[available[-1]]

# Compare no dust vs heavy dust
wavelengths_A = sed.lam.to("Angstrom").value

A_V_values = [0.0, 0.5, 1.0, 2.0]
colours = ['royalblue', 'green', 'orange', 'red']

fig, ax = plt.subplots(figsize=(12, 5))

# Intrinsic (no dust)
ax.loglog(wavelengths_A, sed.luminosity, color='royalblue',
          linewidth=2, label='A_V = 0 (no dust — intrinsic)', alpha=0.9)

# Apply dust manually using the Calzetti law
def apply_calzetti(wavelengths_A, luminosity, A_V):
    w_um = wavelengths_A / 10000
    k = np.where(
        wavelengths_A < 6300,
        2.659 * (-2.156 + 1.509/w_um - 0.198/w_um**2 + 0.011/w_um**3) + 4.05,
        2.659 * (-1.857 + 1.040/w_um) + 4.05
    )
    k = np.clip(k, 0, None)
    return luminosity * 10 ** (-0.4 * A_V * k / 4.05)

lum = sed.luminosity
for A_V, colour in zip([0.5, 1.0, 2.0], ['green', 'orange', 'red']):
    lum_att = apply_calzetti(wavelengths_A, lum, A_V)
    ax.loglog(wavelengths_A, lum_att, color=colour, linewidth=2,
              label=f'A_V = {A_V} mag')

ax.axvspan(3800, 7000, alpha=0.08, color='gold', label='Visible light')
ax.set_xlabel('Wavelength (Å)', fontsize=12)
ax.set_ylabel(r'Luminosity (erg s$^{-1}$ Hz$^{-1}$)', fontsize=12)
ax.set_title(f'Galaxy {GALAXY_ID} — Effect of Dust', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

print("Notice: dust has a much bigger effect at UV (left) than at infrared (right).")
print("This is because dust grains are physically small — about the same size as")
print("UV photons' wavelength — so they scatter/absorb UV much more efficiently.")

### Questions for Experiment 2

1. If a real astronomer measures a galaxy's UV luminosity and uses it to estimate the star formation rate, but does not correct for dust — will they **over-estimate** or **under-estimate** the true rate?

2. The Milky Way has a V-band attenuation of roughly A_V = 0.5 mag toward most of its disk. Compared to the spectrum with no dust, approximately how much UV light is lost?

3. Astronomers often use a quantity called the **UV slope** (how steep the UV spectrum is) as a way to estimate dust. Looking at your plot, does adding more dust make the UV slope steeper or shallower? Why?

---

## Experiment 3 — Compare Different Galaxies

Now let's compare two specific galaxies. They both live in the same TNG50 simulation, but they have different histories — different star formation rates, different amounts of merging, different environments.

In [ ]:
# Download and build a second galaxy for comparison
# This one is a more massive, older 'red and dead' galaxy

GALAXY_ID_2 = 385350   # try also: 468590, 474170, 406595

print(f"Downloading galaxy {GALAXY_ID_2}...")
data_file_2    = download_tng_galaxy(GALAXY_ID_2, api_key=API_KEY)
particles_2    = load_tng_particles(data_file_2)
galaxy_2, grid2 = build_synthesizer_galaxy(
    particles_2,
    grid_dir  = "../grids",
    grid_name = "maraston24-Te00_kroupa-0.1,100"
)
print("\nRunning Synthesizer on galaxy 2...")
sed_2, label_2  = get_spectrum(galaxy_2, grid2)
print("Done!")

In [ ]:
# Compare the spectra of the two galaxies

lam_A = sed.lam.to("Angstrom").value

fig, ax = plt.subplots(figsize=(12, 5))

lum1_norm = sed.luminosity / sed.luminosity.max()
lum2_norm = sed_2.luminosity / sed_2.luminosity.max()

ax.loglog(lam_A, lum1_norm, color='steelblue', lw=2,
          label=f'Galaxy {GALAXY_ID}')
ax.loglog(lam_A, lum2_norm, color='tomato', lw=2,
          label=f'Galaxy {GALAXY_ID_2}')

ax.axvspan(3800, 7000, alpha=0.1, color='gold', label='Visible')
ax.set_xlabel('Wavelength (Å)', fontsize=12)
ax.set_ylabel('Normalised luminosity', fontsize=12)
ax.set_title('Spectral comparison: two TNG50 galaxies', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

In [ ]:
# Compare particle properties side by side

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, p, gid, colour in zip(axes,
    [particles, particles_2],
    [GALAXY_ID, GALAXY_ID_2],
    ['steelblue', 'tomato']):
    ax.hist(p['ages_yr'] / 1e9, bins=40,
            weights=p['masses_msun'], density=True,
            color=colour, alpha=0.7, edgecolor='white', linewidth=0.5)
    ax.axvline(np.average(p['ages_yr']/1e9, weights=p['masses_msun']),
               color='gold', linewidth=2.5, linestyle='--',
               label=f"Mean age = {np.average(p['ages_yr']/1e9, weights=p['masses_msun']):.1f} Gyr")
    ax.set_xlabel('Stellar age (Gyr)', fontsize=12)
    ax.set_ylabel('Mass fraction', fontsize=12)
    ax.set_title(f'Galaxy {gid} — age distribution', fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey question: which galaxy is forming more stars right now?")
print("(A galaxy still forming stars has many young, recently-formed particles.)")

In [ ]:
# Make images of both and compare side-by-side

print("Making images of both galaxies...")
imgs1 = make_image(galaxy,   spectrum_label, fov_kpc=60, pixel_kpc=0.6,
                   filter_codes=["SLOAN/SLOAN.g", "SLOAN/SLOAN.r", "SLOAN/SLOAN.u"])
imgs2 = make_image(galaxy_2, label_2,        fov_kpc=60, pixel_kpc=0.6,
                   filter_codes=["SLOAN/SLOAN.g", "SLOAN/SLOAN.r", "SLOAN/SLOAN.u"])
print("Done!")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6), facecolor='black')

for ax, imgs, gid, title in zip(
    axes,
    [imgs1, imgs2],
    [GALAXY_ID, GALAXY_ID_2],
    [f"Galaxy {GALAXY_ID}", f"Galaxy {GALAXY_ID_2}"]
):
    # Quick single-band display (r-band)
    arr = np.array(imgs["SLOAN/SLOAN.r"].arr)
    arr_log = np.where(arr > 0, np.log10(arr + 1e-40), np.nan)
    vmin = np.nanpercentile(arr_log, 5)
    vmax = np.nanpercentile(arr_log, 99.5)
    ax.imshow(arr_log, origin='lower', cmap='magma', vmin=vmin, vmax=vmax)
    ax.set_title(title + "\n(r-band)", color='white', fontsize=12)
    ax.axis('off')
    ax.set_facecolor('black')

fig.patch.set_facecolor('black')
fig.suptitle("Two TNG50 galaxies compared", color='white', fontsize=14)
plt.tight_layout()
plt.show()

### Questions for Experiment 3

1. Look at the age distributions. Which galaxy is younger on average? Does this match what you see in the spectra?

2. Look at the images. Are the two galaxies the same shape? What might cause differences in morphology (shape)?

3. How would an astronomer classify these two galaxies if they saw them in a real telescope image — as 'blue' or 'red'? What does that tell you about their history?

---

## Experiment 4 — Discovering Galaxy Types

In real astronomy, galaxies are grouped into types based on their appearance and history. The main types are:

| Type | Appearance | Typical history |
|---|---|---|
| **Spiral** | Disk with arms, blue-ish | Actively forming stars; mix of young and old stars |
| **Elliptical** | Smooth, round, reddish | Stopped forming stars long ago; mostly old stars |
| **Irregular** | Patchy, no clear shape | Often small or disturbed by a past collision |
| **Starburst** | Compact, very blue | Extreme burst of star formation happening right now |

Below are four TNG50 galaxies, one of each type. You will download and compare them — what makes each type distinctive in its spectrum, age distribution, and image?

| ID | Expected type |
|---|---|
| `117257` | Large spiral galaxy |
| `468590` | Compact starburst galaxy |
| `385350` | Massive elliptical ('red and dead') |
| `474170` | Dwarf irregular galaxy |

In [ ]:
# Download and process all four galaxy types
# (This will take a few minutes — Synthesizer runs once for each galaxy)

galaxy_types = {
    117257: "Spiral",
    468590: "Starburst",
    385350: "Elliptical",
    474170: "Irregular",
}

type_data = {}

for gal_id, gal_type in galaxy_types.items():
    print(f"\n--- {gal_type} (ID {gal_id}) ---")
    df = download_tng_galaxy(gal_id, api_key=API_KEY)
    p  = load_tng_particles(df)
    g, gr = build_synthesizer_galaxy(
        p,
        grid_dir  = "../grids",
        grid_name = "maraston24-Te00_kroupa-0.1,100"
    )
    s, sl = get_spectrum(g, gr)
    mean_age = np.average(p["ages_yr"] / 1e9, weights=p["masses_msun"])
    type_data[gal_id] = {
        "type":       gal_type,
        "particles":  p,
        "galaxy":     g,
        "sed":        s,
        "spec_label": sl,
        "mass":       p["masses_msun"].sum(),
        "mean_age":   mean_age,
    }
    print(f"  Stellar mass:     {p['masses_msun'].sum():.2e} Msun")
    print(f"  Mean stellar age: {mean_age:.2f} Gyr")

print("\nAll four galaxies ready!")

In [ ]:
# Compare the spectra of all four galaxy types

lam_A = list(type_data.values())[0]["sed"].lam.to("Angstrom").value

type_colours = {
    "Spiral":      "royalblue",
    "Starburst":   "deepskyblue",
    "Elliptical":  "tomato",
    "Irregular":   "mediumpurple",
}

fig, ax = plt.subplots(figsize=(12, 5))

for gal_id, d in type_data.items():
    lum = d["sed"].luminosity
    lum_norm = lum / lum.max()
    ax.loglog(lam_A, lum_norm, linewidth=2,
              color=type_colours[d["type"]],
              label=f'{d["type"]} ({gal_id})')

ax.axvspan(912,   4000,  alpha=0.08, color="violet", label="UV")
ax.axvspan(4000,  7000,  alpha=0.08, color="gold",   label="Visible")
ax.axvspan(7000, 30000,  alpha=0.08, color="tomato",  label="Near-IR")
ax.set_xlabel("Wavelength (Å)", fontsize=12)
ax.set_ylabel("Normalised luminosity", fontsize=12)
ax.set_title("Four galaxy types — spectral comparison", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, which="both")
plt.tight_layout()
plt.show()

print()
print("Which galaxy type is brightest in the UV? Which peaks furthest in the infrared?")
print("What does this tell you about their star formation activity?")

In [ ]:
# Compare stellar age distributions — the star formation history of each galaxy type

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

colour_list = ["royalblue", "deepskyblue", "tomato", "mediumpurple"]

for ax, (gal_id, d), colour in zip(axes, type_data.items(), colour_list):
    p = d["particles"]
    ax.hist(p["ages_yr"] / 1e9, bins=40, weights=p["masses_msun"],
            density=True, color=colour, alpha=0.7,
            edgecolor="white", linewidth=0.4)
    mean_age = d["mean_age"]
    ax.axvline(mean_age, color="gold", linewidth=2.5, linestyle="--",
               label=f"Mean = {mean_age:.1f} Gyr")
    ax.set_xlabel("Stellar age (Gyr)", fontsize=11)
    ax.set_ylabel("Mass fraction", fontsize=11)
    ax.set_title(f'{d["type"]} (ID {gal_id})', fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle("Star formation histories — when were the stars born?", fontsize=13)
plt.tight_layout()
plt.show()

print()
print("The elliptical galaxy formed most of its stars a long time ago (high ages).")
print("The starburst has many very young stars (low ages) — it is forming stars right now.")
print("The spiral has a mix — it has been forming stars slowly over a long time.")

In [ ]:
# Make RGB images of all four galaxy types and compare them in a 2×2 grid!

print("Making images of all four galaxy types...")
for gal_id, d in type_data.items():
    d["images"] = make_image(
        d["galaxy"], d["spec_label"],
        fov_kpc=60, pixel_kpc=0.6,
        filter_codes=["SLOAN/SLOAN.r", "SLOAN/SLOAN.g", "SLOAN/SLOAN.u"],
    )
print("Done!")

def asinh_rgb(imgs, r_filt, g_filt, b_filt, stretch=0.008):
    """Combine three filter images into an RGB picture with astronomer's asinh stretch."""
    r = np.clip(np.array(imgs[r_filt].arr), 0, None)
    g = np.clip(np.array(imgs[g_filt].arr), 0, None)
    b = np.clip(np.array(imgs[b_filt].arr), 0, None)
    I = (r + g + b) / 3.0 + 1e-40
    fac = np.arcsinh(stretch * I) / (stretch * I)
    r_s, g_s, b_s = r * fac, g * fac, b * fac
    top = np.nanpercentile(np.stack([r_s, g_s, b_s]), 99.5)
    return np.clip(np.stack([r_s, g_s, b_s], axis=-1) / top, 0, 1)

fig, axes = plt.subplots(2, 2, figsize=(12, 12), facecolor="black")
axes = axes.flatten()

for ax, (gal_id, d) in zip(axes, type_data.items()):
    rgb = asinh_rgb(d["images"],
                    "SLOAN/SLOAN.r", "SLOAN/SLOAN.g", "SLOAN/SLOAN.u")
    ax.imshow(rgb, origin="lower")
    ax.set_title(f'{d["type"]}\n(ID {gal_id})', color="white", fontsize=12, pad=6)
    ax.axis("off")
    ax.set_facecolor("black")

fig.patch.set_facecolor("black")
fig.suptitle("Four Galaxy Types — RGB Images (r / g / u bands)",
             color="white", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print()
print("Can you see structural differences between the galaxy types?")
print("  Spiral:     look for a disk and extended arms")
print("  Elliptical: smooth, round, concentrated light")
print("  Starburst:  compact, very bright core")
print("  Irregular:  patchy, asymmetric shape")
print()
print("### Questions for Experiment 4")
print()
print("1. Which galaxy looks bluest? Which looks reddest? Does this match what you saw in the spectra?")
print("2. The elliptical galaxy has very little UV emission. Why?")
print("3. If you were an astronomer and could only see this image (no age data),")
print("   how would you try to guess each galaxy's star formation history?")

---

## What You Have Accomplished Today

In one day you have gone from knowing no Python at all to:

- Writing Python code with variables, loops, functions, and plots
- Understanding why astronomers need computers
- Learning what cosmological simulations (IllustrisTNG) are and how they work
- Understanding the concept of **forward modelling**
- Running **Synthesizer** — professional astrophysics software — on real simulation data
- Producing genuine multi-wavelength galaxy images
- Exploring how dust affects what we observe
- Comparing different galaxy types and understanding what makes each one distinct

This is genuinely what research astronomers do — you have just done a miniature version of the work in Sophie's PhD.

---

## Where Next?

If you want to learn more:

- **Python**: *Python Crash Course* by Eric Matthes; free course at https://www.learnpython.org
- **Astronomy**: ESA and NASA websites for current missions; the Hubble and JWST image galleries for real galaxy photographs
- **Synthesizer**: https://docs.synthesizer.space — the full documentation
- **IllustrisTNG**: https://www.tng-project.org — explore all the simulations; you can query galaxy properties directly
---